# Familiarity vs. Answerability: Colab Execution

This notebook orchestrates the preregistered pipeline. It contains no estimators, scientific scoring logic, or claim decisions. Protected endpoints remain closed until their dedicated CLI transactions are available.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN to Colab Secrets before model access."
gpu_query = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip().splitlines()[0]
gpu_name, gpu_memory_mib = [value.strip() for value in gpu_query.rsplit(",", 1)]
disk = shutil.disk_usage("/content")
preflight = {
    "gpu": gpu_name,
    "gpu_gib": round(int(gpu_memory_mib) / 1024, 2),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2),
    "disk_free_gib": round(disk.free / 2**30, 2),
}
assert preflight["gpu_gib"] >= 14 and preflight["disk_free_gib"] >= 40, preflight
preflight


In [ ]:
drive.mount("/content/drive")
REPO = Path("/content/mechanistic-interpretability")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/fa-study-checkpoints")
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
launch_manifest_path = Path(os.environ.get("FA_LAUNCH_MANIFEST", str(DRIVE_CHECKPOINT_ROOT / "fa-study-launch.json")))
launch = json.loads(launch_manifest_path.read_text(encoding="utf-8")) if launch_manifest_path.is_file() else {}
if launch:
    assert set(launch) == {"schema_version", "git_commit", "bundle_file", "bundle_sha256"}
    assert launch["schema_version"] == 1
    assert Path(launch["bundle_file"]).name == launch["bundle_file"]
expected_commit = os.environ.get("FA_GIT_COMMIT") or launch.get("git_commit")
assert expected_commit, "Set FA_GIT_COMMIT to the frozen 40-character commit."
bundle_default = DRIVE_CHECKPOINT_ROOT / launch.get("bundle_file", "fa-study.bundle")
bundle_path = Path(os.environ.get("FA_GIT_BUNDLE", str(bundle_default)))
expected_bundle_sha256 = os.environ.get("FA_GIT_BUNDLE_SHA256") or launch.get("bundle_sha256")
assert bundle_path.is_file(), f"Missing pinned Git bundle: {bundle_path}"
assert expected_bundle_sha256, "Set FA_GIT_BUNDLE_SHA256 before cloning the bundle."
observed_bundle_sha256 = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
assert observed_bundle_sha256 == expected_bundle_sha256, (observed_bundle_sha256, expected_bundle_sha256)
if not REPO.is_dir():
    subprocess.run(["git", "clone", str(bundle_path), str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "--detach", expected_commit], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == expected_commit, (actual_commit, expected_commit)
subprocess.run(["git", "diff", "--quiet"], check=True)
subprocess.run(["git", "diff", "--cached", "--quiet"], check=True)
untracked = subprocess.check_output(["git", "ls-files", "--others", "--exclude-standard"], text=True).splitlines()
unexpected = [path for path in untracked if not path.startswith("runs/familiarity_answerability/")]
assert not unexpected, f"Refusing unexpected untracked files: {unexpected}"
subprocess.run(["git", "bundle", "verify", str(bundle_path)], check=True)


## Pinned environment

Install only the core lock. The optional circuit profile is not part of the confirmatory run.


In [ ]:
assert "torch" not in sys.modules, "Restart the runtime before installing the pinned environment."
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements/fa-core.lock"], check=True)
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
repo_import_path = str((REPO / "src").resolve())
if repo_import_path not in sys.path:
    sys.path.insert(0, repo_import_path)
from importlib import metadata
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

lock_bytes = Path("requirements/fa-core.lock").read_bytes()
lock_requirements = [
    Requirement(line)
    for line in lock_bytes.decode("utf-8").splitlines()
    if line and not line[0].isspace() and not line.startswith("#")
]
lock_names = {canonicalize_name(requirement.name) for requirement in lock_requirements}
lock_mismatches = {
    requirement.name: metadata.version(requirement.name)
    for requirement in lock_requirements
    if not requirement.specifier.contains(
        metadata.version(requirement.name), prereleases=True
    )
}
assert not lock_mismatches, f"Installed versions differ from fa-core.lock: {lock_mismatches}"

pip_check = subprocess.run(
    ["python", "-m", "pip", "check"],
    check=False,
    capture_output=True,
    text=True,
)
pip_check_lines = [line.strip() for line in pip_check.stdout.splitlines() if line.strip()]
locked_conflicts = [
    line
    for line in pip_check_lines
    if canonicalize_name(line.split(" ", 1)[0]) in lock_names
]
assert pip_check.returncode == 0 or pip_check_lines, pip_check.stderr
assert not locked_conflicts, f"Locked dependency conflicts: {locked_conflicts}"
if pip_check.returncode:
    print("Ignored conflicts from Colab-preinstalled packages outside fa-core.lock:")
    print("\n".join(pip_check_lines))
import accelerate
import torch
import transformers
assert torch.__version__.split("+")[0] == "2.7.1", torch.__version__
assert transformers.__version__ == "4.57.1", transformers.__version__
assert accelerate.__version__ == "1.12.0", accelerate.__version__
assert torch.cuda.is_available(), "Pinned PyTorch cannot access the Colab GPU."
gpu = torch.cuda.get_device_properties(0)
preflight["torch_gpu"] = gpu.name
preflight["torch_gpu_gib"] = round(gpu.total_memory / 2**30, 2)
print("fa-core.lock sha256:", hashlib.sha256(lock_bytes).hexdigest())


In [ ]:
CONFIG = "configs/familiarity_answerability_gemma2_2b.json"
ROOT = str(REPO)

def run_cli(*arguments: str, allow_error: bool = False) -> dict:
    command = ["python", "-m", "trajectory_extractor.cli", *arguments]
    print(" ".join(command))
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    if completed.stderr.strip():
        print(completed.stderr)
    print(completed.stdout)
    payload = json.loads(completed.stdout.strip().splitlines()[-1])
    payload["_returncode"] = completed.returncode
    if completed.returncode != 0 and not allow_error:
        raise RuntimeError(f"CLI transaction failed: {payload}")
    return payload


## Frozen Source-v5 factual screening

This phase qualifies the preregistered source pool only. It generates 1,152 factual-screening completions and assembles exactly 244 screened real/synthetic pairs. It does not materialize F1/F2A prompts or open a protected endpoint. Each split is verified and checkpointed independently.


In [ ]:
from trajectory_extractor.fa_artifacts import FAArtifactStore
from trajectory_extractor.fa_colab_checkpoint import ColabSplitCheckpointStore
from trajectory_extractor.fa_config import FAConfig

SOURCE_ROOT = REPO / "data/fa/confirmatory_source_v5"
SOURCE_INTEGRITY = REPO / "data/fa/confirmatory_source_v5/source_integrity_v1.json"
config = FAConfig.from_json(REPO / CONFIG)
store = FAArtifactStore(REPO)
SCREENING_SPLITS = {
    "mechanism_train": {
        "candidates": SOURCE_ROOT / "candidate_entities_mechanism_train_v1.json",
        "questions": SOURCE_ROOT / "screening_questions_mechanism_train_v1.json",
        "synthetic": SOURCE_ROOT / "synthetic_candidates_mechanism_train_v1.json",
        "completion_count": 384,
        "match_count": 80,
    },
    "locked_validation": {
        "candidates": SOURCE_ROOT / "candidate_entities_locked_validation_v1.json",
        "questions": SOURCE_ROOT / "screening_questions_locked_validation_v1.json",
        "synthetic": SOURCE_ROOT / "synthetic_candidates_locked_validation_v1.json",
        "completion_count": 192,
        "match_count": 40,
    },
    "behavior_test": {
        "candidates": SOURCE_ROOT / "candidate_entities_behavior_test_v1.json",
        "questions": SOURCE_ROOT / "screening_questions_behavior_test_v1.json",
        "synthetic": SOURCE_ROOT / "synthetic_candidates_behavior_test_v1.json",
        "completion_count": 288,
        "match_count": 60,
    },
    "probe_test": {
        "candidates": SOURCE_ROOT / "candidate_entities_probe_test_v1.json",
        "questions": SOURCE_ROOT / "screening_questions_probe_test_v1.json",
        "synthetic": SOURCE_ROOT / "synthetic_candidates_probe_test_v1.json",
        "completion_count": 144,
        "match_count": 32,
    },
    "intervention_test": {
        "candidates": SOURCE_ROOT / "candidate_entities_intervention_test_v1.json",
        "questions": SOURCE_ROOT / "screening_questions_intervention_test_v1.json",
        "synthetic": SOURCE_ROOT / "synthetic_candidates_intervention_test_v1.json",
        "completion_count": 144,
        "match_count": 32,
    },
}
assert sum(value["completion_count"] for value in SCREENING_SPLITS.values()) == 1152
assert sum(value["match_count"] for value in SCREENING_SPLITS.values()) == 244
assert SOURCE_INTEGRITY.is_file()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

identity_dir = DRIVE_CHECKPOINT_ROOT / actual_commit
identity_dir.mkdir(parents=True, exist_ok=True)
identity = {
    "git_commit": actual_commit,
    "bundle_sha256": expected_bundle_sha256,
    "launch_manifest_sha256": sha256_file(launch_manifest_path) if launch else None,
    "lock_sha256": sha256_file(REPO / "requirements/fa-core.lock"),
    "config_sha256": config.config_hash,
    "source_integrity_sha256": sha256_file(SOURCE_INTEGRITY),
    "model_id": config.model_id,
    "model_revision": config.model_revision,
    "tokenizer_revision": config.tokenizer_revision,
    "chat_template_sha256": config.chat_template_sha256,
}
identity_bytes = (json.dumps(identity, indent=2, sort_keys=True) + "\n").encode("utf-8")
identity_path = identity_dir / "execution_identity.json"
if identity_path.exists():
    assert identity_path.read_bytes() == identity_bytes, "Execution identity changed."
else:
    temporary_identity = identity_path.with_suffix(".json.partial")
    temporary_identity.write_bytes(identity_bytes)
    os.replace(temporary_identity, identity_path)
runtime_observation = {
    **preflight,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
}
runtime_bytes = (json.dumps(runtime_observation, indent=2, sort_keys=True) + "\n").encode("utf-8")
runtime_sha256 = hashlib.sha256(runtime_bytes).hexdigest()
runtime_path = identity_dir / f"runtime_observation-{runtime_sha256[:16]}.json"
if runtime_path.exists():
    assert runtime_path.read_bytes() == runtime_bytes
else:
    runtime_partial = runtime_path.with_suffix(".json.partial")
    runtime_partial.write_bytes(runtime_bytes)
    os.replace(runtime_partial, runtime_path)

checkpoint_store = ColabSplitCheckpointStore(
    repo_root=REPO,
    checkpoint_root=identity_dir,
    scratch_root=Path("/content"),
    run_id=config.run_id,
    git_commit=actual_commit,
    config_sha256=config.config_hash,
)


In [ ]:
screened_manifests = []
for split, spec in SCREENING_SPLITS.items():
    checkpoint_store.restore_split_checkpoint(split)
    screened_manifest = checkpoint_store.unique_manifest(split, "screened_match")
    if screened_manifest is None:
        completion_manifest = checkpoint_store.successful_completion_manifest(split)
        if completion_manifest is None:
            generated = run_cli(
                "fa-run-screening",
                "--config", CONFIG,
                "--root", ROOT,
                "--namespace", split,
                "--candidates-manifest", str(spec["candidates"]),
                "--questions-manifest", str(spec["questions"]),
                "--source-integrity-manifest", str(SOURCE_INTEGRITY),
                "--shard-id", checkpoint_store.next_screening_shard_id(split),
                allow_error=True,
            )
            if generated["_returncode"] != 0:
                if generated.get("shard_manifest"):
                    store.verify_shard(generated["shard_manifest"])
                    checkpoint_store.checkpoint_split(split, "failure")
                raise RuntimeError(f"Screening generation failed for {split}: {generated}")
            assert generated["status"] == "generated", generated
            assert generated["count"] == spec["completion_count"], generated
            completion_manifest = Path(generated["shard_manifest"])
            completion_shard = store.verify_shard(completion_manifest)
            assert completion_shard.row_count == spec["completion_count"]
            checkpoint_store.checkpoint_split(split, "completion")
        screened = run_cli(
            "fa-screen-entities",
            "--config", CONFIG,
            "--root", ROOT,
            "--candidates-manifest", str(spec["candidates"]),
            "--questions-manifest", str(spec["questions"]),
            "--screening-manifest", str(completion_manifest),
            "--synthetic-manifest", str(spec["synthetic"]),
            "--source-integrity-manifest", str(SOURCE_INTEGRITY),
            allow_error=True,
        )
        if screened["_returncode"] != 0:
            checkpoint_store.checkpoint_split(split, "completion")
            raise RuntimeError(f"Entity screening failed for {split}: {screened}")
        assert screened["status"] == "screened", screened
        assert screened["count"] == spec["match_count"], screened
        store.verify_shard(screened["audit_manifest"])
        screened_manifest = Path(screened["manifest"])
        assert store.verify_shard(screened_manifest).row_count == spec["match_count"]
    checkpoint_store.checkpoint_split(split, "screened")
    screened_manifests.append(screened_manifest)

assembly_arguments = [
    "fa-assemble-screened-matches",
    "--config", CONFIG,
    "--root", ROOT,
]
for manifest in screened_manifests:
    assembly_arguments.extend(["--screened-matches-manifest", str(manifest)])
assembly_arguments.extend(["--shard-id", "confirmatory-screened-collection-v1"])
assembled = run_cli(*assembly_arguments)
assert assembled["_returncode"] == 0, assembled
assert assembled["status"] == "assembled", assembled
assert assembled["count"] == 244, assembled
collection_manifest = Path(assembled["manifest"])
assert store.verify_shard(collection_manifest).row_count == 244
checkpoint_store.checkpoint_split("mechanism_train", "collection")
print("Source-v5 screening complete:", collection_manifest)


## Deliberate protocol stop

Do not continue into task construction or protected F1/F2A execution. First export the blinded naturalness packets and obtain two independent human ratings, with a third independent adjudicator available for disagreements.


In [ ]:
STOP_AFTER_SCREENING_ASSEMBLY = True
assert not STOP_AFTER_SCREENING_ASSEMBLY, (
    "Intentional stop: complete the preregistered two-rater naturalness gate before later cells."
)


## Resumable core sequence

Set the paths below only to verified artifacts from the same run. `fa-materialize-probe-rows` creates compact, provenance-bound evidence from generation, registered activations, exact teacher-forced scores, metadata, and outcomes. Protected rows remain unreadable to selection code.


In [ ]:
run_cli("fa-audit-manifest", "--config", CONFIG, "--root", ROOT, "--manifest", "<verified-manifest>")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "mechanism_train", "--manifest", "<mechanism-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "mechanism-0000", "--resume")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "locked_validation", "--manifest", "<validation-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "validation-0000", "--resume")
run_cli("fa-fit-probes", "--config", CONFIG, "--root", ROOT, "--train-rows-manifest", "<mechanism-probe-rows>", "--validation-rows-manifest", "<validation-probe-rows>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--shard-id", "selection-0000")
run_cli("fa-seal-behavior-test", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>")
run_cli("fa-seal-selection", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>")
run_cli("fa-evaluate-behavior-test", "--config", CONFIG, "--root", ROOT, "--manifest", "<behavior-test-prompt-manifest>", "--shard-id", "behavior-0000")
run_cli("fa-evaluate-probe-test", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "probe-test-0000")
run_cli("fa-build-report", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--selection-manifest", "<f2a-selection-manifest>", "--output", "reports/familiarity_answerability.md")


## Stop conditions

Stop on any pin, hash, audit, OOM, completion, or endpoint-state failure. Run transactions on local Colab storage. After each successful transaction, checkpoint only completed checksum-verified shards to `DRIVE_CHECKPOINT_ROOT`; restore only artifacts that pass their sidecar verification. A runtime interruption resumes from verified shard manifests rather than filenames. The notebook does not treat Google Drive writes as atomic.
